# Lab 1: End-to-End VarNet for Accelerated MRI Reconstruction

In this lab you will explore how a learned reconstruction model (VarNet) recovers MRI images from undersampled k-space data. You will:

1. Visualize fully-sampled k-space and the effect of undersampling masks
2. Run a pretrained VarNet on 4x-accelerated data and measure image quality (SSIM)
3. Investigate what happens when the undersampling pattern at test time differs from training
4. Ablate the data-consistency (DC) module to understand its role
5. Push the model to 20x acceleration and examine clinical implications

**Prerequisites:** If you're seeing this notebook, you should already be on a GPU node with Jupyter running. If not, follow the setup instructions in the README.

## Part 0 — Setup & Data Exploration

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd
import torch

from utils.transforms import (
    to_tensor, fft2c, ifft2c, complex_abs, rss, complex_center_crop,
    RandomMaskFunc, EquispacedMaskFractionFunc, apply_mask,
)
from utils.metrics import ssim_metric
from models.varnet import SimpleVarNet

# ---------- paths (edit if needed) ----------
SCRATCH   = "/gpfs/scratch/johnsp23/DLrecon_lab1"
DATA_DIR  = os.path.join(SCRATCH, "data", "knee")
SPLIT_CSV = os.path.join(SCRATCH, "data", "fastMRI_paired_knee.csv")
CKPT_DIR  = os.path.join(SCRATCH, "pretrained")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

### Dataset overview

The fastMRI knee dataset contains multicoil k-space data stored as HDF5 files. Each file has:
- `kspace`: shape `(slices, 15, 640, 368)` — 15 coils, 640 readout lines, 368 phase-encode lines (complex64)
- `reconstruction_rss`: shape `(slices, 320, 320)` — fully-sampled root-sum-of-squares target

The dataset includes two MRI contrasts acquired in the same session for each patient:
- **PD** (proton density) — higher intrinsic SNR
- **PDFS** (proton density fat-suppressed) — fat signal is suppressed, better for visualizing pathology near fat

Both contrasts are used for training and evaluation. Let's check the split sizes and load examples of each.

In [ ]:
# Load split CSV and count files per split (both contrasts)
df = pd.read_csv(SPLIT_CSV)

print("Volumes per split (PD + PDFS contrasts):")
print(f"{'split':>5s}  {'PD':>4s}  {'PDFS':>4s}  {'total':>5s}")
print("-" * 28)
for split in ["train", "val", "test"]:
    n_pd   = df[df["pd_split"] == split].shape[0]
    n_pdfs = df[df["pdfs_split"] == split].shape[0]
    print(f"{split:>5s}  {n_pd:4d}  {n_pdfs:4d}  {n_pd + n_pdfs:5d}")

# Build test file lists for each contrast (we'll use these throughout the lab)
# Use 10 volumes per contrast for faster inference
N_EVAL = 10
test_files_pd   = df.loc[df["pd_split"] == "test", "pd"].dropna().tolist()[:N_EVAL]
test_files_pdfs = df.loc[df["pdfs_split"] == "test", "pdfs"].dropna().tolist()[:N_EVAL]
test_files      = list(set(test_files_pd + test_files_pdfs))
print(f"\nTest files: {len(test_files_pd)} PD + {len(test_files_pdfs)} PDFS = {len(test_files)} total")

In [ ]:
# Load one PD volume for exploration
example_file_pd = os.path.join(DATA_DIR, test_files_pd[0])
with h5py.File(example_file_pd, "r") as f:
    kspace_vol = f["kspace"][:]
    target_vol = f["reconstruction_rss"][:]
    print(f"PD file: {test_files_pd[0]}")
    print(f"kspace shape:  {kspace_vol.shape}  dtype: {kspace_vol.dtype}")
    print(f"target shape:  {target_vol.shape}  dtype: {target_vol.dtype}")

# Pick a mid-volume slice for visualization
sl = kspace_vol.shape[0] // 2
kspace_slice = kspace_vol[sl]   # (coils, 640, 368) complex64
target_slice = target_vol[sl]   # (320, 320)

# Also load a PDFS volume to compare contrasts
example_file_pdfs = os.path.join(DATA_DIR, test_files_pdfs[0])
with h5py.File(example_file_pdfs, "r") as f:
    target_vol_pdfs = f["reconstruction_rss"][:]
    print(f"PDFS file: {test_files_pdfs[0]}")

sl_pdfs = target_vol_pdfs.shape[0] // 2
target_slice_pdfs = target_vol_pdfs[sl_pdfs]

In [ ]:
# Compare PD vs PDFS contrasts
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(target_slice, cmap="gray")
axes[0].set_title("PD (proton density)")
axes[0].axis("off")
axes[1].imshow(target_slice_pdfs, cmap="gray")
axes[1].set_title("PDFS (PD fat-suppressed)")
axes[1].axis("off")
plt.suptitle("Two contrasts in the dataset", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### Visualize k-space: magnitude and phase

K-space is the raw frequency-domain data acquired by the MRI scanner. Each coil produces its own k-space. Below we show:
- **Log-magnitude** of k-space (note the bright center — low frequencies carry most of the signal energy)
- **Phase** of k-space

In [ ]:
# K-space visualization: pick coil 0
coil0 = kspace_slice[0]  # (640, 368) complex64

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Log-magnitude
log_mag = np.log1p(np.abs(coil0))
axes[0].imshow(log_mag, cmap="viridis", aspect="auto")
axes[0].set_title("K-space log-magnitude (coil 0)")
axes[0].set_xlabel("Phase-encode")
axes[0].set_ylabel("Readout")

# Phase
axes[1].imshow(np.angle(coil0), cmap="twilight", aspect="auto")
axes[1].set_title("K-space phase (coil 0)")
axes[1].set_xlabel("Phase-encode")

plt.tight_layout()
plt.show()

In [ ]:
# Fully-sampled image: coil images and RSS combination
kspace_t = to_tensor(kspace_slice)  # (coils, 640, 368, 2)
image_t = ifft2c(kspace_t)          # (coils, 640, 368, 2)
image_crop = complex_center_crop(image_t, (320, 320))
coil_images = complex_abs(image_crop)  # (coils, 320, 320)
rss_image = rss(coil_images, dim=0)    # (320, 320)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(3):
    axes[i].imshow(coil_images[i].numpy(), cmap="gray")
    axes[i].set_title(f"Coil {i}")
    axes[i].axis("off")
axes[3].imshow(rss_image.numpy(), cmap="gray")
axes[3].set_title("RSS combination")
axes[3].axis("off")
plt.suptitle("Individual coil images and RSS combination", y=1.02)
plt.tight_layout()
plt.show()

### Undersampling masks

MRI acceleration works by acquiring only a fraction of the k-space lines. The **mask** determines which lines are sampled. We compare two strategies:
- **Random**: center lines are always fully acquired (ACS region), then `1/R` of the remaining outer lines are randomly sampled
- **Equispaced**: center lines + every Rth line across the full k-space

Note: the acceleration factor `R` applies to the **outer** k-space lines only. Because the center (ACS) lines are always fully sampled on top, the effective acceleration is less than `R`. For example, with `R=4` and 8% ACS on 368 PE lines: 29 center + 85 outer = 114 lines sampled, giving an effective acceleration of ~3.2x (31% of lines).

In [ ]:
# Generate and visualize random vs equispaced masks
random_mask_func = RandomMaskFunc(center_fractions=[0.08], accelerations=[4])
equi_mask_func   = EquispacedMaskFractionFunc(center_fractions=[0.08], accelerations=[4])

shape = (1, 640, 368, 2)  # (coils, H, W, 2)
random_mask, _ = random_mask_func(shape, seed=42)
equi_mask, _   = equi_mask_func(shape, seed=42)

# Masks are shape (1, 1, 368, 1) — squeeze to 1D for plotting
rm_1d = random_mask.squeeze().numpy()
em_1d = equi_mask.squeeze().numpy()

# Print effective acceleration
print(f"Random mask:     {rm_1d.sum():.0f}/{len(rm_1d)} lines sampled "
      f"(effective acceleration: {len(rm_1d)/rm_1d.sum():.1f}x)")
print(f"Equispaced mask: {em_1d.sum():.0f}/{len(em_1d)} lines sampled "
      f"(effective acceleration: {len(em_1d)/em_1d.sum():.1f}x)")

# Show masks as 2D (replicated across readout)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(np.tile(rm_1d, (50, 1)), cmap="gray", aspect="auto")
axes[0].set_title(f"Random mask ({rm_1d.sum():.0f}/{len(rm_1d)} lines, ~{len(rm_1d)/rm_1d.sum():.1f}x)")
axes[0].set_xlabel("Phase-encode")

axes[1].imshow(np.tile(em_1d, (50, 1)), cmap="gray", aspect="auto")
axes[1].set_title(f"Equispaced mask ({em_1d.sum():.0f}/{len(em_1d)} lines, ~{len(em_1d)/em_1d.sum():.1f}x)")
axes[1].set_xlabel("Phase-encode")

plt.tight_layout()
plt.show()

In [ ]:
# Zero-filled reconstructions: what does undersampled data look like without a model?
masked_random, mask_r, _ = apply_mask(kspace_t, random_mask_func, seed=42)
masked_equi, mask_e, _   = apply_mask(kspace_t, equi_mask_func, seed=42)

def zero_filled_recon(masked_kspace):
    """IFFT -> crop -> RSS of undersampled k-space."""
    img = ifft2c(masked_kspace)
    img = complex_center_crop(img, (320, 320))
    return rss(complex_abs(img), dim=0)

zf_random = zero_filled_recon(masked_random)
zf_equi   = zero_filled_recon(masked_equi)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(target_slice, cmap="gray")
axes[0].set_title("Fully sampled")
axes[1].imshow(zf_random.numpy(), cmap="gray")
axes[1].set_title("Zero-filled: Random 4x")
axes[2].imshow(zf_equi.numpy(), cmap="gray")
axes[2].set_title("Zero-filled: Equispaced 4x")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## Part 1 — Inference: Random 4x Model

We load a VarNet pretrained on **random 4x** undersampled data and run inference on the test set using both random and equispaced masks. This reveals how the model performs when the test-time mask distribution matches or differs from training.

In [ ]:
# Helper: load a pretrained model
def load_model(ckpt_path, use_dc=True):
    """Load a SimpleVarNet checkpoint."""
    ckpt = torch.load(ckpt_path, map_location="cpu")
    cfg = ckpt["config"]
    model = SimpleVarNet(
        num_cascades=cfg.get("num_cascades", 12),
        chans=cfg.get("chans", 24),
        pools=cfg.get("pools", 4),
        sens_chans=cfg.get("sens_chans", 8),
        sens_pools=cfg.get("sens_pools", 4),
        use_dc=cfg.get("use_dc", use_dc),
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model.to(device)

# Load the random 4x model
model_random4x = load_model(os.path.join(CKPT_DIR, "varnet_random4x.pt"))
print(f"Loaded varnet_random4x.pt (val SSIM at training: {torch.load(os.path.join(CKPT_DIR, 'varnet_random4x.pt'), map_location='cpu').get('val_ssim', 'N/A')})")

In [ ]:
# Helper: run inference on one slice
@torch.no_grad()
def run_inference(model, kspace_np, mask_func, seed=None):
    """
    Run VarNet inference on a single k-space slice.
    
    Returns: (reconstruction, masked_kspace, mask, num_low_freqs)
    All returned as CPU tensors.
    """
    kspace = to_tensor(kspace_np)  # (coils, H, W, 2)
    masked_kspace, mask, num_low_freqs = apply_mask(kspace, mask_func, seed=seed)
    
    # Add batch dim, move to device
    output = model(
        masked_kspace.unsqueeze(0).to(device),
        mask.unsqueeze(0).to(device),
        num_low_frequencies=int(num_low_freqs),
    )
    return output.squeeze(0).cpu(), masked_kspace, mask, num_low_freqs


SKIP_SLICES = 4  # Skip first N slices per volume (noisy edge slices, not knee joint)

@torch.no_grad()
def eval_volume(model, kspace_vol, target_vol, max_value, mask_func, seed=None, batch_size=8):
    """
    Run batched inference on an entire volume and return per-slice SSIM values.
    Skips the first SKIP_SLICES slices (mostly noise, not useful anatomy).
    All slices share the same mask (same seed), so we can batch efficiently.
    """
    # Generate mask once from the first slice
    kspace0 = to_tensor(kspace_vol[0])
    _, mask, num_low_freqs = apply_mask(kspace0, mask_func, seed=seed)
    
    num_slices = kspace_vol.shape[0]
    ssim_vals = []
    
    for start in range(SKIP_SLICES, num_slices, batch_size):
        end = min(start + batch_size, num_slices)
        batch_kspace = to_tensor(kspace_vol[start:end])           # (B, coils, H, W, 2)
        batch_masked = batch_kspace * mask.unsqueeze(0) + 0.0     # broadcast mask
        batch_target = torch.from_numpy(target_vol[start:end].astype(np.float32))
        
        recons = model(
            batch_masked.to(device),
            mask.unsqueeze(0).to(device),
            num_low_frequencies=int(num_low_freqs),
        ).cpu()
        
        for i in range(recons.shape[0]):
            ssim_vals.append(ssim_metric(recons[i], batch_target[i], max_value=max_value))
    
    return ssim_vals

In [ ]:
# Evaluate on test set with RANDOM and EQUISPACED 4x masks
random_mask_func_eval = RandomMaskFunc(center_fractions=[0.08], accelerations=[4])
equi_mask_func_eval   = EquispacedMaskFractionFunc(center_fractions=[0.08], accelerations=[4])

# Track SSIM separately for PD and PDFS contrasts
test_files_pd_set   = set(test_files_pd)
test_files_pdfs_set = set(test_files_pdfs)

ssim_random = []
ssim_equi   = []
ssim_random_pd = []
ssim_random_pdfs = []
ssim_equi_pd = []
ssim_equi_pdfs = []

for i, fname in enumerate(test_files):
    fpath = os.path.join(DATA_DIR, fname)
    with h5py.File(fpath, "r") as f:
        kspace_vol = f["kspace"][:]
        target_vol = f["reconstruction_rss"][:]
        max_value  = float(f.attrs["max"])
    
    seed = tuple(map(ord, fname))
    is_pd   = fname in test_files_pd_set
    is_pdfs = fname in test_files_pdfs_set
    
    # Random mask — batched
    vals_r = eval_volume(model_random4x, kspace_vol, target_vol, max_value,
                         random_mask_func_eval, seed=seed)
    ssim_random.extend(vals_r)
    if is_pd:
        ssim_random_pd.extend(vals_r)
    if is_pdfs:
        ssim_random_pdfs.extend(vals_r)
    
    # Equispaced mask — batched
    vals_e = eval_volume(model_random4x, kspace_vol, target_vol, max_value,
                         equi_mask_func_eval, seed=seed)
    ssim_equi.extend(vals_e)
    if is_pd:
        ssim_equi_pd.extend(vals_e)
    if is_pdfs:
        ssim_equi_pdfs.extend(vals_e)
    
    print(f"\r  {i+1}/{len(test_files)}", end="", flush=True)

print()
print(f"Random 4x model on RANDOM 4x masks:     SSIM = {np.mean(ssim_random):.4f} +/- {np.std(ssim_random):.4f}")
print(f"  PD  slices:  SSIM = {np.mean(ssim_random_pd):.4f} +/- {np.std(ssim_random_pd):.4f}  (n={len(ssim_random_pd)})")
print(f"  PDFS slices: SSIM = {np.mean(ssim_random_pdfs):.4f} +/- {np.std(ssim_random_pdfs):.4f}  (n={len(ssim_random_pdfs)})")
print(f"\nRandom 4x model on EQUISPACED 4x masks:  SSIM = {np.mean(ssim_equi):.4f} +/- {np.std(ssim_equi):.4f}")
print(f"  PD  slices:  SSIM = {np.mean(ssim_equi_pd):.4f} +/- {np.std(ssim_equi_pd):.4f}  (n={len(ssim_equi_pd)})")
print(f"  PDFS slices: SSIM = {np.mean(ssim_equi_pdfs):.4f} +/- {np.std(ssim_equi_pdfs):.4f}  (n={len(ssim_equi_pdfs)})")

In [ ]:
# Visualize example slices: ground truth | reconstruction | error map
# Re-run on the example slice we loaded earlier
recon_random, _, _, _ = run_inference(model_random4x, kspace_slice, random_mask_func_eval, seed=42)
recon_equi, _, _, _   = run_inference(model_random4x, kspace_slice, equi_mask_func_eval, seed=42)

error_scale = 2  # amplify error for visibility

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Random mask
axes[0, 0].imshow(target_slice, cmap="gray")
axes[0, 0].set_title("Ground truth")
axes[0, 1].imshow(recon_random.numpy(), cmap="gray")
axes[0, 1].set_title(f"Random 4x (SSIM={ssim_metric(recon_random, torch.from_numpy(target_slice), max_value=float(target_slice.max())):.4f})")
err_r = np.abs(recon_random.numpy() - target_slice)
axes[0, 2].imshow(err_r * error_scale, cmap="gray", vmin=0, vmax=target_slice.max() * 0.1)
axes[0, 2].set_title(f"Error x{error_scale}")

# Row 2: Equispaced mask
axes[1, 0].imshow(target_slice, cmap="gray")
axes[1, 0].set_title("Ground truth")
axes[1, 1].imshow(recon_equi.numpy(), cmap="gray")
axes[1, 1].set_title(f"Equispaced 4x (SSIM={ssim_metric(recon_equi, torch.from_numpy(target_slice), max_value=float(target_slice.max())):.4f})")
err_e = np.abs(recon_equi.numpy() - target_slice)
axes[1, 2].imshow(err_e * error_scale, cmap="gray", vmin=0, vmax=target_slice.max() * 0.1)
axes[1, 2].set_title(f"Error x{error_scale}")

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

### Question 1 (write your answer below)

Compare the SSIM values and error maps for the random vs. equispaced masks. 
- Why does the model perform differently on the two mask types?
- What do you notice about the spatial pattern of the errors in each case?

Also look at the PD vs. PDFS SSIM breakdown above.
- Which contrast achieves higher SSIM? Why might that be the case?

*Your answer here.*

---
## Part 2 — Inference: No Data Consistency

VarNet alternates between a learned regularizer (U-Net) and a **data consistency (DC)** step that enforces agreement with the acquired k-space measurements. Here we load a model trained *without* DC (`use_dc=False`) to understand what DC contributes.

Without DC, the model is a pure image-processing pipeline — it cannot "look back" at the raw measurements to correct itself.

In [ ]:
# Load the no-DC model
model_nodc = load_model(os.path.join(CKPT_DIR, "varnet_random4x_noDC.pt"), use_dc=False)

# Evaluate on test set with random 4x mask
ssim_nodc = []
for i, fname in enumerate(test_files):
    fpath = os.path.join(DATA_DIR, fname)
    with h5py.File(fpath, "r") as f:
        kspace_vol = f["kspace"][:]
        target_vol = f["reconstruction_rss"][:]
        max_value  = float(f.attrs["max"])
    
    seed = tuple(map(ord, fname))
    ssim_nodc.extend(eval_volume(model_nodc, kspace_vol, target_vol, max_value,
                                  random_mask_func_eval, seed=seed))
    print(f"\r  {i+1}/{len(test_files)}", end="", flush=True)

print()
print(f"Random 4x WITH DC:    SSIM = {np.mean(ssim_random):.4f} +/- {np.std(ssim_random):.4f}")
print(f"Random 4x WITHOUT DC: SSIM = {np.mean(ssim_nodc):.4f} +/- {np.std(ssim_nodc):.4f}")
print(f"DC improvement:       {np.mean(ssim_random) - np.mean(ssim_nodc):+.4f}")

In [ ]:
# Side-by-side comparison on the example slice
recon_nodc, _, _, _ = run_inference(model_nodc, kspace_slice, random_mask_func_eval, seed=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

target_t = torch.from_numpy(target_slice)
mv = float(target_slice.max())

# Row 1: With DC
axes[0, 0].imshow(target_slice, cmap="gray")
axes[0, 0].set_title("Ground truth")
axes[0, 1].imshow(recon_random.numpy(), cmap="gray")
axes[0, 1].set_title(f"With DC (SSIM={ssim_metric(recon_random, target_t, max_value=mv):.4f})")
axes[0, 2].imshow(np.abs(recon_random.numpy() - target_slice) * error_scale, cmap="gray",
                   vmin=0, vmax=target_slice.max() * 0.1)
axes[0, 2].set_title(f"Error x{error_scale}")

# Row 2: Without DC
axes[1, 0].imshow(target_slice, cmap="gray")
axes[1, 0].set_title("Ground truth")
axes[1, 1].imshow(recon_nodc.numpy(), cmap="gray")
axes[1, 1].set_title(f"Without DC (SSIM={ssim_metric(recon_nodc, target_t, max_value=mv):.4f})")
axes[1, 2].imshow(np.abs(recon_nodc.numpy() - target_slice) * error_scale, cmap="gray",
                   vmin=0, vmax=target_slice.max() * 0.1)
axes[1, 2].set_title(f"Error x{error_scale}")

for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].set_ylabel("With DC", fontsize=14)
axes[1, 0].set_ylabel("Without DC", fontsize=14)
plt.tight_layout()
plt.show()

### Question 2 (write your answer below)

Does reconstruction (with data consistency) give better image quality than post-processing (without data consistency)? What evidence do you see in the SSIM values and error maps?

---
## Part 3 — Inference: 20x Acceleration

What happens when we push acceleration much further? With `R=20` and 4% ACS on 368 PE lines, only 33 lines are sampled (~11x effective acceleration, 9% of k-space). We load a VarNet trained at this setting and examine both general image quality and a specific clinical case: a **meniscal tear**.

The key question: at extreme acceleration, does the model preserve clinically important pathology, or does it smooth it away?

In [ ]:
# Load 20x model
model_20x = load_model(os.path.join(CKPT_DIR, "varnet_random20x.pt"))

# 20x mask function
mask_func_20x = RandomMaskFunc(center_fractions=[0.04], accelerations=[20])

# Evaluate on test set
ssim_20x = []
for i, fname in enumerate(test_files):
    fpath = os.path.join(DATA_DIR, fname)
    with h5py.File(fpath, "r") as f:
        kspace_vol = f["kspace"][:]
        target_vol = f["reconstruction_rss"][:]
        max_value  = float(f.attrs["max"])
    
    seed = tuple(map(ord, fname))
    ssim_20x.extend(eval_volume(model_20x, kspace_vol, target_vol, max_value,
                                 mask_func_20x, seed=seed))
    print(f"\r  {i+1}/{len(test_files)}", end="", flush=True)

print()
print(f"Random  4x model: SSIM = {np.mean(ssim_random):.4f} +/- {np.std(ssim_random):.4f}")
print(f"Random 20x model: SSIM = {np.mean(ssim_20x):.4f} +/- {np.std(ssim_20x):.4f}")

In [ ]:
# General example: compare 4x vs 20x on our example slice
recon_20x_ex, _, _, _ = run_inference(model_20x, kspace_slice, mask_func_20x, seed=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(target_slice, cmap="gray")
axes[0].set_title("Ground truth")
axes[1].imshow(recon_random.numpy(), cmap="gray")
axes[1].set_title(f"VarNet 4x (SSIM={ssim_metric(recon_random, target_t, max_value=mv):.4f})")
axes[2].imshow(recon_20x_ex.numpy(), cmap="gray")
axes[2].set_title(f"VarNet 20x (SSIM={ssim_metric(recon_20x_ex, target_t, max_value=mv):.4f})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Pathology case: meniscal tear

Below we examine specific slices known to contain meniscal pathology. The question is whether the tear remains visible after 20x reconstruction, or if the model hallucinates a "normal" meniscus.

In [ ]:
# Pathology case: meniscal tear
fname, sl = "file1000942.h5", 24
fpath = os.path.join(DATA_DIR, fname)

with h5py.File(fpath, "r") as f:
    ks = f["kspace"][sl]
    tgt = f["reconstruction_rss"][sl]
    mv_path = float(f.attrs["max"])

tgt_t = torch.from_numpy(tgt.astype(np.float32))
seed = tuple(map(ord, fname))

recon_4x, _, _, _ = run_inference(model_random4x, ks, random_mask_func_eval, seed=seed)
recon_20x, _, _, _ = run_inference(model_20x, ks, mask_func_20x, seed=seed)

# Crop to right half, vertically centered (zoom into patient-left meniscus)
crop = np.s_[80:240, 160:320]
vmax = tgt[crop].max() * 0.5  # brighten to make meniscal tear more visible

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(tgt[crop], cmap="gray", vmin=0, vmax=vmax)
axes[0].set_title(f"Ground truth\n{fname} slice {sl}")
axes[1].imshow(recon_4x.numpy()[crop], cmap="gray", vmin=0, vmax=vmax)
axes[1].set_title(f"VarNet 4x\nSSIM={ssim_metric(recon_4x, tgt_t, max_value=mv_path):.4f}")
axes[2].imshow(recon_20x.numpy()[crop], cmap="gray", vmin=0, vmax=vmax)
axes[2].set_title(f"VarNet 20x\nSSIM={ssim_metric(recon_20x, tgt_t, max_value=mv_path):.4f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Question 3 (write your answer below)

- Compare the 4x and 20x reconstructions on the pathology slices. Is the meniscal tear still visible at 20x?
- What are the clinical implications of using aggressive acceleration in diagnostic MRI?
- SSIM measures overall image similarity. Why might SSIM be insufficient for evaluating clinical image quality?

*Your answer here.*

---
## Summary

In [ ]:
# Summary table of all experiments
print(f"{'Experiment':<45s}  {'SSIM (mean +/- std)':>20s}")
print("-" * 68)
print(f"{'Random 4x model, random 4x mask':<45s}  {np.mean(ssim_random):.4f} +/- {np.std(ssim_random):.4f}")
print(f"{'  -> PD only':<45s}  {np.mean(ssim_random_pd):.4f} +/- {np.std(ssim_random_pd):.4f}")
print(f"{'  -> PDFS only':<45s}  {np.mean(ssim_random_pdfs):.4f} +/- {np.std(ssim_random_pdfs):.4f}")
print(f"{'Random 4x model, equispaced 4x mask':<45s}  {np.mean(ssim_equi):.4f} +/- {np.std(ssim_equi):.4f}")
print(f"{'No-DC model, random 4x mask':<45s}  {np.mean(ssim_nodc):.4f} +/- {np.std(ssim_nodc):.4f}")
print(f"{'Random 20x model, random 20x mask':<45s}  {np.mean(ssim_20x):.4f} +/- {np.std(ssim_20x):.4f}")

---
## Homework

### Option A: Train an equispaced 4x model

The pretrained models above were all trained with **random** undersampling. Your task: train a VarNet on **equispaced 4x** data and compare.

1. Create a new config file `configs/equispaced_4x.yaml` (copy `random_4x.yaml` and change `mask_type: equispaced`)
2. Submit a training job using the provided script:

```bash
# Example SLURM submission script (adjust partition/GPU as needed):

#!/bin/bash
#SBATCH --job-name=vn_equi4x
#SBATCH --partition=a100_short
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=2-00:00:00
#SBATCH --output=runs/equi4x_%j.out

conda activate /gpfs/scratch/johnsp23/DLrecon_lab1/envs/varnet
cd ~/dl-mri-recon-lab

python scripts/train.py \\
    --config configs/equispaced_4x.yaml \\
    --data_path /gpfs/scratch/johnsp23/DLrecon_lab1/data/knee \\
    --split_csv /gpfs/scratch/johnsp23/DLrecon_lab1/data/fastMRI_paired_knee.csv \\
    --output_dir runs/equi4x
```

3. After training, evaluate your checkpoint on the test set using `scripts/eval.py`:

```bash
# Evaluate your equispaced-trained model:
python scripts/eval.py --checkpoint runs/equi4x/best.pt \\
    --data_path /gpfs/scratch/johnsp23/DLrecon_lab1/data/knee \\
    --split_csv /gpfs/scratch/johnsp23/DLrecon_lab1/data/fastMRI_paired_knee.csv

# Also evaluate with random masks to see cross-mask generalization:
python scripts/eval.py --checkpoint runs/equi4x/best.pt \\
    --data_path /gpfs/scratch/johnsp23/DLrecon_lab1/data/knee \\
    --split_csv /gpfs/scratch/johnsp23/DLrecon_lab1/data/fastMRI_paired_knee.csv \\
    --mask_type random

# Save example comparison figures:
python scripts/eval.py --checkpoint runs/equi4x/best.pt \\
    --data_path /gpfs/scratch/johnsp23/DLrecon_lab1/data/knee \\
    --split_csv /gpfs/scratch/johnsp23/DLrecon_lab1/data/fastMRI_paired_knee.csv \\
    --save_figures figures/equi4x
```

The eval script reads model config directly from the checkpoint, so it uses the same config you trained with. Use `--mask_type` to override the mask and test cross-mask generalization.

4. Compare SSIM to the random-trained model evaluated on equispaced masks (from Part 1). Comment on the difference.

### Option B (Advanced): Extend to 3D

The current model processes each slice independently. Extend it to use 3D information:
- Consider using **slice-unique random masks** (different random mask per slice in a volume)
- Or **shifted equispaced masks** (offset the equispaced pattern by a different amount per slice)
- How does incorporating cross-slice information change reconstruction quality?